# ADAS-Vision — Google Colab T4 Full Pipeline (A → Z)

Notebook tổng hợp chạy trọn vẹn trên **Google Colab (T4 GPU)**:
1. Kiểm tra GPU & Mount Google Drive
2. Clone/mở dự án & cài đặt thư viện
3. Giải nén dữ liệu gốc BDD100K & Cityscapes vào SSD `/content/datasets` (tốc độ cao)
4. Khảo sát nhãn thật (`inspect_bdd.py`) & tạo split + nhãn YOLO (`prepare_data.py`)
5. Huấn luyện Object Detection **YOLO11** (`detector_best.pt`)
6. Huấn luyện Semantic Segmentation **DeepLabV3+** (`segmenter_best.pt`)
7. Đánh giá định lượng (`evaluate.py`: mAP, IoU, Dice, Cityscapes, FPS 720p) & Xuất Video Demo HUD (`run_inference.py`)

## Cell 1 — Kiểm tra GPU T4 & Kết nối Google Drive

In [1]:
!nvidia-smi

from google.colab import drive

drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/adas_outputs/weights
!mkdir -p /content/drive/MyDrive/adas_outputs/reports
!mkdir -p /content/drive/MyDrive/adas_outputs/videos

Thu Nov 26 03:34:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Tải Mã Nguồn Dự Án & Cài Đặt Thư Viện
Điền URL GitHub repo của nhóm (hoặc bỏ comment dòng `%cd` nếu để code trực tiếp trên Google Drive).

In [2]:

!git clone https://github.com/dainguyenbao2006-cyber/Developing-utility-vehicles-and-regional-transit-networks.git
!git switch tv1
%cd Developing-utility-vehicles-and-regional-transit-networks/

# Cài đặt thư viện

!pip install -q ultralytics segmentation-models-pytorch albumentations \
    pyyaml cityscapesscripts kagglehub

Cloning into 'Developing-utility-vehicles-and-regional-transit-networks'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 221 (delta 63), reused 203 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 469.74 KiB | 9.99 MiB/s, done.
Resolving deltas: 100% (63/63), done.
fatal: not a git repository (or any of the parent directories): .git
/content/Developing-utility-vehicles-and-regional-transit-networks
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [16]:
!git switch tv1

branch 'tv1' set up to track 'origin/tv1'.
Switched to a new branch 'tv1'


In [17]:
!git status

On branch tv1
Your branch is up to date with 'origin/tv1'.

nothing to commit, working tree clean


## Cell 3 — Tải Trực Tiếp BDD100K Vào SSD `/content/datasets` Qua KaggleHub
Mirror `dl.cv.ethz.ch` cũ đã ngừng hoạt động. Cell này dùng `kagglehub` tải thẳng bản gốc `solesensei/solesensei_bdd100k` (gồm 100K ảnh + JSON chứa đủ cả `box2d`, `drivable area` và `lane`) vào ổ SSD của Colab mà **không tốn dung lượng Google Drive**.

In [18]:
from pathlib import Path

import kagglehub

bdd_path = Path(kagglehub.dataset_download('solesensei/solesensei_bdd100k'))
print('BDD100K đã tải xong tại:', bdd_path)

Using Colab cache for faster access to the 'solesensei_bdd100k' dataset.
BDD100K đã tải xong tại: /kaggle/input/solesensei_bdd100k


## Cell 4 — Khảo Sát Dữ Liệu Thật & Sinh Nhãn YOLO + Mask Segmentation (`seed=42`)

In [19]:
import os

os.environ["BDD_ROOT"] = str(bdd_path)
os.environ["BDD_WORK_DIR"] = "/content/bdd_prepared"
os.environ["CITYSCAPES_ROOT"] = "/content/datasets/cityscapes"
os.environ["DEVICE"] = "cuda"

# 1. Kiểm tra thực tế cấu trúc thư mục và JSON categories
!python scripts/inspect_bdd.py \
    --save-json /content/drive/MyDrive/adas_outputs/reports/bdd_inspection.json

# 2. Sinh split cố định (seed=42), nhãn YOLO .txt VÀ tự động vẽ mask drivable/lane từ JSON!
# Gợi ý: 20,000 train / 2,000 val chạy vừa khít phiên Colab T4 miễn phí.
!python scripts/prepare_data.py --max-train 20000 --max-val 2000

2026-11-26 03:50:09,801 | INFO | BDD100K Inspection Summary:
{
  "bdd_root": "/kaggle/input/solesensei_bdd100k",
  "subdirs_depth_3": [
    "bdd100k",
    "bdd100k/bdd100k",
    "bdd100k/bdd100k/images",
    "bdd100k_labels_release",
    "bdd100k_labels_release/bdd100k",
    "bdd100k_labels_release/bdd100k/labels",
    "bdd100k_seg",
    "bdd100k_seg/bdd100k",
    "bdd100k_seg/bdd100k/seg"
  ],
  "image_counts": {},
  "json_files_found": [
    "/kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json",
    "/kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json"
  ],
  "json_reports": [
    {
      "path": "/kaggle/input/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json",
      "schema_type": "list",
      "total_frames_in_file": 69863,
      "inspected_frames": 5000,
      "categories": {
        "bike": 527,
        "bus": 836,
        "car": 51388,
  

## Cell 5 — Huấn Luyện Object Detection (YOLO11) & Benchmark

In [20]:
# Huấn luyện mô hình chính YOLO11s -> lưu trực tiếp sang Google Drive
!python scripts/train_detector.py \
    --data configs/bdd_yolo.yaml \
    --model yolo11s.pt \
    --epochs 30 \
    --batch-size 16 \
    --device 0 \
    --output-weights /content/drive/MyDrive/adas_outputs/weights/detector_best.pt
# (Tùy chọn) Huấn luyện thêm YOLO11n / YOLOv8s để lập bảng đối sánh Benchmark:
# !python scripts/train_detector.py --data configs/bdd_yolo.yaml --model yolo11n.pt \
#     --epochs 30 --batch-size 32 --device 0 \
#     --output-weights /content/drive/MyDrive/adas_outputs/weights/yolo11n_best.pt

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
2026-11-26 03:55:04,544 | INFO | Initializing YOLO model: yolo11s.pt
2026-11-26 03:55:05,050 | INFO | Starting training with arguments:
{'data': 'configs/bdd_yolo.yaml', 'epochs': 30, 'batch': 16, 'imgsz': 640, 'optimizer': 'AdamW', 'lr0': 0.001, 'workers': 4, 'amp': True, 'close_mosaic': 10, 'fliplr': 0.5, 'flipud': 0.0, 'project': 'runs/detect', 'name': 'yolo11_bdd', 'exist_ok': True, 'device': '0'}
Ultralytics 8.4.219 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_rem

In [9]:
!cp /content/drive/MyDrive/adas_outputs/weights/detector_best.pt weights/detector_best.pt

cp: cannot stat '/content/drive/MyDrive/adas_outputs/weights/detector_best.pt': No such file or directory


## Cell 6 — Huấn Luyện Semantic Segmentation (DeepLabV3+)

In [21]:
# Nếu kết quả inspect_bdd.py ở Cell 4 báo pixel ID khác mặc định (0, 1, 255),
# hãy truyền thêm --direct-id <id> --alt-id <id> --lane-bg-id <id>
!python scripts/train_segmenter.py \
    --epochs 15 \
    --batch-size 16 \
    --device cuda \
    --output-weights /content/drive/MyDrive/adas_outputs/weights/segmenter_best.pt



2026-11-26 07:05:36,398 | INFO | Using device: cuda
2026-11-26 07:05:47,817 | INFO | Initialized BDDSegmentationDataset with 19958 samples from /content/bdd_prepared/yolo/images/train
2026-11-26 07:05:48,879 | INFO | Initialized BDDSegmentationDataset with 2000 samples from /content/bdd_prepared/yolo/images/val
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-11-26 07:05:48,880 | INFO | Initializing DeepLabV3Plus (encoder=resnet34, classes=3)...
2026-11-26 07:05:49,315 | INFO | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
202

In [12]:
!cp /content/drive/MyDrive/adas_outputs/weights/segmenter_best.pt weights/segmenter_best.pt

cp: cannot stat '/content/drive/MyDrive/adas_outputs/weights/segmenter_best.pt': No such file or directory


## Cell 7 — Đánh Giá Toàn Diện (BDD100K + Cityscapes + FPS T4) & Xuất Video Demo

In [22]:
# Khôi phục weights từ Drive (hữu ích khi mở phiên Colab mới chỉ để đánh giá/xuất video)
!cp /content/drive/MyDrive/adas_outputs/weights/detector_best.pt weights/detector_best.pt
!cp /content/drive/MyDrive/adas_outputs/weights/segmenter_best.pt weights/segmenter_best.pt

# 1. Xuất báo cáo thực nghiệm định lượng (JSON + bảng Markdown)
!python scripts/evaluate.py \
    --mode all \
    --device cuda \
    --output-json /content/drive/MyDrive/adas_outputs/reports/eval_report.json \
    --output-md /content/drive/MyDrive/adas_outputs/reports/evaluation_report.md

# Hiển thị ngay bảng kết quả Markdown vừa đo được
!cat /content/drive/MyDrive/adas_outputs/reports/evaluation_report.md

2026-11-26 08:16:22,720 | INFO | Loading YOLO model from: weights/detector_best.pt
2026-11-26 08:16:23,011 | INFO | Loading segmentation checkpoint from: weights/segmenter_best.pt
2026-11-26 08:16:23,686 | INFO | Running validation benchmark on data=configs/bdd_yolo.yaml with imgsz=640...
Ultralytics 8.4.219 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,416,283 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 92.8±31.2 MB/s, size: 52.8 KB)
val: Scanning /content/bdd_prepared/yolo/labels/val.cache... 2000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2000/2000 335.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 250/250 10.5it/s 23.9s
                   all       2000      36669      0.739      0.425      0.391      0.233
Speed: 0.7ms preprocess, 5.1ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved t

In [23]:
# 2. Xuất video demo có vẽ HUD cảnh báo ADAS (đặt sẵn clip .mp4 vào MyDrive/adas_data/)
for clip in ["day_clear", "night_drive", "rain_drive", "vn_traffic"]:
    src = f"/content/drive/MyDrive/adas_data/{clip}.mp4"
    out_vid = f"/content/drive/MyDrive/adas_outputs/videos/{clip}_adas_hud.mp4"
    out_csv = f"/content/drive/MyDrive/adas_outputs/videos/{clip}_events.csv"
    print(f"=== Đang xử lý {clip}.mp4 ===")
    !python scripts/run_inference.py run \
        --source "{src}" \
        --output "{out_vid}" \
        --log-csv "{out_csv}" \
        --max-frames 450 \
        --device cuda \
        --no-display

=== Đang xử lý day_clear.mp4 ===
2026-11-26 08:18:50,311 | INFO | Loading YOLO model from: weights/detector_best.pt
2026-11-26 08:18:50,674 | INFO | Loading segmentation checkpoint from: weights/segmenter_best.pt
2026-11-26 08:19:42,750 | INFO | Processed 450 frames in 49.75s (avg 9.05 FPS).
2026-11-26 08:19:42,750 | INFO | Completed run: 450 frames | Avg FPS: 9.05 | Video: /content/drive/MyDrive/adas_outputs/videos/day_clear_adas_hud.mp4 | Events CSV: /content/drive/MyDrive/adas_outputs/videos/day_clear_events.csv
=== Đang xử lý night_drive.mp4 ===
2026-11-26 08:19:50,168 | INFO | Loading YOLO model from: weights/detector_best.pt
2026-11-26 08:19:50,491 | INFO | Loading segmentation checkpoint from: weights/segmenter_best.pt
2026-11-26 08:20:20,518 | INFO | Processed 280 frames in 28.78s (avg 9.73 FPS).
2026-11-26 08:20:20,518 | INFO | Completed run: 280 frames | Avg FPS: 9.73 | Video: /content/drive/MyDrive/adas_outputs/videos/night_drive_adas_hud.mp4 | Events CSV: /content/drive/MyD

In [24]:
# Nén thư mục dữ liệu đã prepare thành file zip và lưu trực tiếp vào Google Drive
!zip -r /content/drive/MyDrive/adas_outputs/bdd_prepared.zip /content/bdd_prepared

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  adding: content/bdd_prepared/yolo/images/train/930d648a-f2b79e81.jpg (deflated 2%)
  adding: content/bdd_prepared/yolo/images/train/66c50638-78a279bf.jpg (deflated 2%)
  adding: content/bdd_prepared/yolo/images/train/0b91b26d-2d0f5e80.jpg (deflated 2%)
  adding: content/bdd_prepared/yolo/images/train/1fb24e18-3fda319d.jpg (deflated 2%)
  adding: content/bdd_prepared/yolo/images/train/01c4035b-bcaeb067.jpg (deflated 1%)
  adding: content/bdd_prepared/yolo/images/train/411d00f9-bcd61a00.jpg (deflated 2%)
  adding: content/bdd_prepared/yolo/images/train/453e1100-6e5623e1.jpg (deflated 11%)
  adding: content/bdd_prepared/yolo/images/train/7ffaf795-9835a113.jpg (deflated 1%)
  adding: content/bdd_prepared/yolo/images/train/27af15fa-cc1cf7a9.jpg (deflated 3%)
  adding: content/bdd_prepared/yolo/images/train/245ddac9-3a37c8d4.jpg (deflated 8%)
  adding: content/bdd_prepared/yolo/images/train/08c7ed37-e58f9ddf.jpg (deflated 5%)
  addin